In [39]:
!pip install google-play-scraper
!pip install gensim

# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews, Sort, reviews_all
import time

import pandas as pd  # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None  # Menonaktifkan peringatan chaining
import numpy as np  # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed)  # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt  # Matplotlib untuk visualisasi data
import seaborn as sns  # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score

import datetime as dt  # Manipulasi data waktu dan tanggal
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt_tab') # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.
nltk.download('wordnet') # Mengunduh dataset WordNet untuk lemmatization.
nltk.download('omw-1.4') # Mengunduh Open Multilingual WordNet untuk mendukung WordNet.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 19.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [23]:
#membaca file ulasan aplikasi dari folder
import pandas as pd
df = pd.read_csv('ulasan_aplikasi.csv')  # Membaca file CSV yang berisi ulasan aplikasi dan menyimpannya dalam DataFrame bernama 'df'.


# Loading Dataset

In [24]:
df.head()

,Review
0,Tolong beri fitur nonaktifkan notifikasi promo...
1,Saya cukup kecewa sebagai user yang baru kemba...
2,"jelek, sekarang dapet driver jadi lama tidak k..."
3,"sering banget susah dapet driver, bahkan selal..."
4,"aneh banget jaringan bagus, buka aplikasi lain..."


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  15000 non-null  object
dtypes: object(1)
memory usage: 117.3+ KB


In [ ]:
# Membuat DataFrame baru (clean_df) dengan menghapus baris yang memiliki nilai yang hilang (NaN) dari fitur review
clean_df = df.dropna()

In [34]:
# Menghapus baris duplikat dari DataFrame clean_df
clean_df = clean_df.drop_duplicates()

# Menghitung jumlah baris dan kolom dalam DataFrame clean_df setelah menghapus duplikat
jumlah_ulasan_setelah_hapus_duplikat, jumlah_kolom_setelah_hapus_duplikat = clean_df.shape

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  15000 non-null  object
dtypes: object(1)
memory usage: 117.3+ KB


In [36]:
df.head()

,Review
0,Tolong beri fitur nonaktifkan notifikasi promo...
1,Saya cukup kecewa sebagai user yang baru kemba...
2,"jelek, sekarang dapet driver jadi lama tidak k..."
3,"sering banget susah dapet driver, bahkan selal..."
4,"aneh banget jaringan bagus, buka aplikasi lain..."


# Preprocessing Text

In [40]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # Memecah teks menjadi daftar kata
    words = text.split()

    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]

    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [41]:
#penghapus kumpulan slang words atau kata-kata informal yang sering digunakan dalam percakapan sehari-hari
slangwords = slangwords = {
    "jg": "juga", 
    "dgn": "dengan", 
    "udh": "sudah",
    "sdh": "sudah", 
    "blm": "belum", 
    "tp": "tapi",
    "kalo": "kalau", 
    "gk": "tidak", 
    "ga": "tidak",
    "gak": "tidak", 
    "aja": "saja", 
    "dr": "dari", 
    "sy": "saya", 
    "gw": "saya",
    "bgt": "banget",
    "utk": "untuk",
    "krn": "karena"
}
def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text


In [42]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['Review'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

In [43]:
import csv
import requests
from io import StringIO

# Membaca data kamus kata-kata positif dari GitHub
lexicon_positive = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub

if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma

    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_positive[row[0]] = int(row[1])
        # Menambahkan kata-kata positif dan skornya ke dalam kamus lexicon_positive
else:
    print("Failed to fetch positive lexicon data")

# Membaca data kamus kata-kata negatif dari GitHub
lexicon_negative = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub

if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma

    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_negative[row[0]] = int(row[1])
        # Menambahkan kata-kata negatif dan skornya dalam kamus lexicon_negative
else:
    print("Failed to fetch negative lexicon data")

In [44]:
# Fungsi untuk menentukan polaritas sentimen dari tweet

def sentiment_analysis_lexicon_indonesia(text):
    #for word in text:

    score = 0
    # Inisialisasi skor sentimen ke 0

    for word in text:
        # Mengulangi setiap kata dalam teks

        if (word in lexicon_positive):
            score = score + lexicon_positive[word]
            # Jika kata ada dalam kamus positif, tambahkan skornya ke skor sentimen

    for word in text:
        # Mengulangi setiap kata dalam teks (sekali lagi)

        if (word in lexicon_negative):
            score = score + lexicon_negative[word]
            # Jika kata ada dalam kamus negatif, kurangkan skornya dari skor sentimen

    polarity=''
    # Inisialisasi variabel polaritas

    if (score > 0):
        polarity = 'positive'
        # Jika skor sentimen lebih besar atau sama dengan 0, maka polaritas adalah positif
    elif (score < 0):
        polarity = 'negative'
        # Jika skor sentimen kurang dari 0, maka polaritas adalah negatif

    else:
        polarity = 'neutral'
    # Ini adalah bagian yang bisa digunakan untuk menentukan polaritas netral jika diperlukan

    return score, polarity
    # Mengembalikan skor sentimen dan polaritas teks

In [45]:
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]
print(clean_df['polarity'].value_counts())

polarity
negative    9027
positive    5137
neutral      830
Name: count, dtype: int64


In [46]:
clean_df.head()

,Review,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir,polarity_score,polarity
0,Tolong beri fitur nonaktifkan notifikasi promo...,Tolong beri fitur nonaktifkan notifikasi promo...,tolong beri fitur nonaktifkan notifikasi promo...,tolong beri fitur nonaktifkan notifikasi promo...,"[tolong, beri, fitur, nonaktifkan, notifikasi,...","[tolong, fitur, nonaktifkan, notifikasi, promo...",tolong fitur nonaktifkan notifikasi promo meng...,-2,negative
1,Saya cukup kecewa sebagai user yang baru kemba...,Saya cukup kecewa sebagai user yang baru kemba...,saya cukup kecewa sebagai user yang baru kemba...,saya cukup kecewa sebagai user yang baru kemba...,"[saya, cukup, kecewa, sebagai, user, yang, bar...","[kecewa, user, gojek, fitur, beli, barang, apk...",kecewa user gojek fitur beli barang apk gojek ...,9,positive
2,"jelek, sekarang dapet driver jadi lama tidak k...",jelek sekarang dapet driver jadi lama tidak ko...,jelek sekarang dapet driver jadi lama tidak ko...,jelek sekarang dapet driver jadi lama tidak ko...,"[jelek, sekarang, dapet, driver, jadi, lama, t...","[jelek, dapet, driver, konsisten, driver, posi...",jelek dapet driver konsisten driver posisinya ...,-12,negative
3,"sering banget susah dapet driver, bahkan selal...",sering banget susah dapet driver bahkan selalu...,sering banget susah dapet driver bahkan selalu...,sering banget susah dapet driver bahkan selalu...,"[sering, banget, susah, dapet, driver, bahkan,...","[banget, susah, dapet, driver, dapet, udah, ba...",banget susah dapet driver dapet udah banget nu...,0,neutral
4,"aneh banget jaringan bagus, buka aplikasi lain...",aneh banget jaringan bagus buka aplikasi lain ...,aneh banget jaringan bagus buka aplikasi lain ...,aneh banget jaringan bagus buka aplikasi lain ...,"[aneh, banget, jaringan, bagus, buka, aplikasi...","[aneh, banget, jaringan, bagus, buka, aplikasi...",aneh banget jaringan bagus buka aplikasi amana...,-11,negative


# Skenario 1

## Pelatihan: SVM,    Ekstraksi Fitur: TF-IDF,    Pembagian Data: 80/20

In [47]:
from sklearn.model_selection import train_test_split

# Pisahkan data menjadi fitur (tweet) dan label (sentimen)
X = clean_df['text_akhir']
y = clean_df['polarity']

# Mapping sederhana
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
y = clean_df['polarity'].map(label_map)


In [48]:
# Bagi data menjadi data latih dan data uji dengan data traning 80 dan testing 20
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.2, random_state=42)

In [ ]:
#menggunakan metode smote untuk mengatasi masalah ketidakseimbangan kelas dalam data latih
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from collections import Counter

# Vektorisasi TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
X_tfidf_train = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Jalankan SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_tfidf_train, y_train)

print("Distribusi kelas sekarang seimbang:", Counter(y_resampled))

Distribusi kelas sekarang seimbang: Counter({2: 7234, 0: 7234, 1: 7234})


In [50]:
pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 11.3 MB/s eta 0:00:00


In [51]:
#mencari parameter terbaik untuk model SVM menggunakan Bayesian Optimization dengan BayesSearchCV dari scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.svm import SVC

# 1. Perkecil ruang pencarian
search_space = {
    'C': Real(0.1, 100, prior='log-uniform'),   # Rentang lebih masuk akal
    'gamma': Real(0.01, 1, prior='log-uniform'),
    'kernel': Categorical(['linear', 'rbf'])    # Hapus 'poly' karena terlalu berat
}

# 2. Sesuaikan setting pencarian
opt = BayesSearchCV(
    estimator=SVC(),
    search_spaces=search_space,
    n_iter=10,
    cv=3,           # Ubah ke 3-fold (Mengurangi total training dari 50 jadi 30 kali)
    n_jobs=-1,
    verbose=3,      # Ubah ke 3 agar Anda bisa melihat pesan detail setiap detik
    random_state=42,
    scoring='f1_weighted'
)

# 3. Jalankan proses pencarian pada data yang sudah di-SMOTE
# Ingat: Gunakan X_train_resampled dan y_train_resampled dari langkah sebelumnya
opt.fit(X_resampled, y_resampled)

# 4. Hasil Terbaik
print("Akurasi Terbaik: %s" % opt.best_score_)
print("Parameter Terbaik: %s" % opt.best_params_)

# 5. Evaluasi pada data testing
y_pred = opt.predict(X_test_tfidf)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Akurasi Terbaik: 0.9226710883156599
Parameter Terbaik: OrderedDict({'C': 73.52481813242629, 'gamma': 0.2519085390684862, 'kernel': 'rbf'})


In [79]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
# Menginisialisasi algoritma Support Vector Classifier (SVC)
svm = SVC(C=73.52481813242629, gamma=0.2519085390684862, kernel='rbf', random_state=42, class_weight='balanced')
#melakukan training model
svm.fit(X_resampled, y_resampled)


SVC(C=73.52481813242629, class_weight='balanced', gamma=0.2519085390684862,
    random_state=42)

In [80]:
#menghitung akurasi model
y_pred = svm.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
accuracy


0.8432810936978993

# Skenario 2
## Pelatihan: RF,    Ekstraksi Fitur: Word2Vec,    Pembagian Data: 80/20

In [56]:
from gensim.models import Word2Vec
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [57]:
# Pembagian Data: 80% Training, 20% Testing
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [58]:
# Tokenisasi: Pecah kalimat menjadi list kata
sentences = [sentence.split() for sentence in X_train2]

# Training Model Word2Vec
# vector_size=100 berarti setiap kata diwakili 100 koordinat angka
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Fungsi untuk merubah satu kalimat menjadi satu vektor rata-rata
def get_sentence_vector(sentence, model):
    words = sentence.split()
    # Ambil vektor kata yang ada di kamus Word2Vec, lalu rata-ratakan
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    if len(word_vectors) > 0:
        return np.mean(word_vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

# 4. Transformasi X_train dan X_test menjadi vektor
# Gunakan fungsi get_sentence_vector yang sudah Anda buat
X_train_w2v = np.array([get_sentence_vector(s, w2v_model) for s in X_train2])
X_test_w2v = np.array([get_sentence_vector(s, w2v_model) for s in X_test2])

In [59]:
# Jalankan SMOTE pada vektor Word2Vec
smote = SMOTE(random_state=42)
X_resampled2, y_resampled2 = smote.fit_resample(X_train_w2v, y_train2)


In [60]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical
from sklearn.ensemble import RandomForestClassifier

# 1. Tentukan Ruang Pencarian (Search Space)
# Kita fokus pada parameter yang paling berpengaruh pada performa dan waktu
search_space_rf = {
    'n_estimators': Integer(100, 500),         # Jumlah pohon
    'max_depth': Integer(10, 50),              # Kedalaman maksimal pohon
    'min_samples_split': Integer(2, 10),       # Sampel minimal untuk split simpul
    'min_samples_leaf': Integer(1, 5),         # Sampel minimal di setiap daun
    'max_features': Categorical(['sqrt', 'log2']) # Jumlah fitur tiap pohon
}

# 2. Inisialisasi Bayesian Search
opt_rf = BayesSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    search_spaces=search_space_rf,
    n_iter=15,               # 15 iterasi cukup untuk awal di dataset 15k
    cv=3,                    # 3-fold CV agar proses lebih cepat
    n_jobs=-1,               # Gunakan semua core CPU Fedora Anda
    verbose=3,               # Menampilkan log progress secara detail
    random_state=42,
    scoring='f1_weighted'
)

# 3. Jalankan Proses Training
# Pastikan menggunakan data yang sudah di-SMOTE dan dipisah secara benar
opt_rf.fit(X_resampled2, y_resampled2)

# 4. Output Hasil Terbaik
print("Skor F1 Terbaik: %s" % opt_rf.best_score_)
print("Parameter Terbaik: %s" % opt_rf.best_params_)
                      

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Skor F1 Terbaik: 0.8256716732859428
Parameter Terbaik: OrderedDict({'max_depth': 50, 'max_features': 'sqrt', 'min_s

In [62]:
#Hasil Terbaik
print("Akurasi Terbaik: %s" % opt_rf.best_score_)
print("Parameter Terbaik: %s" % opt_rf.best_params_)

Akurasi Terbaik: 0.8256716732859428
Parameter Terbaik: OrderedDict({'max_depth': 50, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500})


In [63]:
#menghitung dengan algoritma random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
# Menginisialisasi algoritma Support Vector Classifier (SVC)
rf_model = RandomForestClassifier(n_estimators=500, max_depth=50, min_samples_split=2, min_samples_leaf=1, max_features='sqrt', random_state=42)
#melakukan training model
rf_model.fit(X_resampled2, y_resampled2)

RandomForestClassifier(max_depth=50, n_estimators=500, random_state=42)

In [64]:
# 4. Evaluasi
y_pred2 = rf_model.predict(X_test_w2v)
accuracy = accuracy_score(y_test2, y_pred2)
accuracy

0.6305435145048349

# Skenario 3
## Pelatihan: RF,    Ekstraksi Fitur: TF-IDF,    Pembagian Data: 70/30  

In [68]:
# Bagi data menjadi data latih dan data uji
X_train3, X_test3, y_train3, y_test3 = train_test_split(X,y , test_size=0.3, random_state=42)

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE

#Ubah TEXT (X) menjadi NUMERIC (Vektor)
tfidf2 = TfidfVectorizer(max_features=1000)
X__train_tfidf2 = tfidf2.fit_transform(X_train3)
X_test_tfidf2 = tfidf2.transform(X_test3)

# Jalankan SMOTE (Sekarang X dan y sudah numerik)
smote2 = SMOTE(random_state=42)
X_resampled3, y_resampled3 = smote2.fit_resample(X__train_tfidf2, y_train3)

print("Distribusi kelas sekarang seimbang:", Counter(y_resampled3))


Distribusi kelas sekarang seimbang: Counter({0: 6327, 2: 6327, 1: 6327})


In [70]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical
from sklearn.ensemble import RandomForestClassifier

# 1. Tentukan Ruang Pencarian (Search Space)
# Kita fokus pada parameter yang paling berpengaruh pada performa dan waktu
search_space_rf = {
    'n_estimators': Integer(100, 500),         # Jumlah pohon
    'max_depth': Integer(10, 50),              # Kedalaman maksimal pohon
    'min_samples_split': Integer(2, 10),       # Sampel minimal untuk split simpul
    'min_samples_leaf': Integer(1, 5),         # Sampel minimal di setiap daun
    'max_features': Categorical(['sqrt', 'log2']) # Jumlah fitur tiap pohon
}

# 2. Inisialisasi Bayesian Search
opt_rf2 = BayesSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    search_spaces=search_space_rf,
    n_iter=10,               # 10 iterasi cukup untuk awal di dataset 15k
    cv=3,                    # 3-fold CV agar proses lebih cepat
    n_jobs=-1,               # Gunakan semua core CPU Fedora Anda
    verbose=3,               # Menampilkan log progress secara detail
    random_state=42,
    scoring='f1_weighted'
)

# 3. Jalankan Proses Training
# Pastikan menggunakan data yang sudah di-SMOTE dan dipisah secara benar
opt_rf2.fit(X_resampled3, y_resampled3)

# 4. Output Hasil Terbaik
print("Skor F1 Terbaik: %s" % opt_rf2.best_score_)
print("Parameter Terbaik: %s" % opt_rf2.best_params_)
                      

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Skor F1 Terbaik: 0.8026031818231395
Parameter Terbaik: OrderedDict({'max_depth': 28, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 175})


In [71]:
rf_model2 = RandomForestClassifier(n_estimators=175, max_depth=28, min_samples_split=5, min_samples_leaf=1, max_features='log2', random_state=42)
rf_model2.fit(X_resampled3, y_resampled3)

RandomForestClassifier(max_depth=28, max_features='log2', min_samples_split=5,
                       n_estimators=175, random_state=42)

In [72]:
#Evaluasi
y_pred3 = rf_model2.predict(X_test_tfidf2)
accuracy = accuracy_score(y_test3, y_pred3)
accuracy

0.6972660591242499

#  inference atau testing Model

In [74]:
#untuk skenario 1
#teks yang akan di uji
test_texts = [
    "Aplikasi Gojek sangat membantu saya memesan makanan dengan cepat!",
    "Biasa saja, tidak ada yang istimewa dari update kali ini.",
    "Kecewa banget, aplikasinya sering error dan driver tidak datang.",
    "Aplikasi ini sangat jelek sekali",
    "Ada beberapa promo yang muncul di halaman utama tadi pagi."
]

#Transformasi teks ke bentuk numerik
X_new_tfidf = tfidf.transform(test_texts)

#Prediksi menggunakan model SVM terbaik hasil RandomSearch/BayesSearch
y_pred_new = svm.predict(X_new_tfidf)

# Mapping hasil angka kembali ke label teks
label_map_inverse = {0: 'negatif', 1: 'netral', 2: 'positif'}
results = [label_map_inverse[pred] for pred in y_pred_new]

# menampilkan hasil
print("-" * 50)
print("HASIL INFERENSI MODEL SVM")
print("-" * 50)
for teks, label in zip(test_texts, results):
    print(f"Teks  : {teks}")
    print(f"Hasil : {label.upper()}")
    print("-" * 50)

--------------------------------------------------
HASIL INFERENSI MODEL SVM
--------------------------------------------------
Teks  : Aplikasi Gojek sangat membantu saya memesan makanan dengan cepat!
Hasil : NEGATIF
--------------------------------------------------
Teks  : Biasa saja, tidak ada yang istimewa dari update kali ini.
Hasil : NETRAL
--------------------------------------------------
Teks  : Kecewa banget, aplikasinya sering error dan driver tidak datang.
Hasil : NEGATIF
--------------------------------------------------
Teks  : Aplikasi ini sangat jelek sekali
Hasil : NEGATIF
--------------------------------------------------
Teks  : Ada beberapa promo yang muncul di halaman utama tadi pagi.
Hasil : NETRAL
--------------------------------------------------


In [76]:
#skenario 2
#teks yang akan di uji
test_texts = [
    "Aplikasi Gojek sangat membantu saya memesan makanan dengan cepat!",
    "Biasa saja, tidak ada yang istimewa dari update kali ini.",
    "Kecewa banget, aplikasinya sering error dan driver tidak datang.",
    "Aplikasi ini sangat jelek sekali",
    "Ada beberapa promo yang muncul di halaman utama tadi pagi."
]

# Transformasi teks ke bentuk numerik (Wajib pakai tfidf yang sudah di-fit sebelumnya)
X_new_tfidf2 = np.array([get_sentence_vector(s, w2v_model) for s in test_texts])

# Prediksi menggunakan model SVM terbaik hasil RandomSearch/BayesSearch
y_pred_new = rf_model.predict(X_new_tfidf2)

# Mapping hasil angka kembali ke label teks
label_map_inverse = {0: 'negatif', 1: 'netral', 2: 'positif'}
results = [label_map_inverse[pred] for pred in y_pred_new]

#menampilkan hasil
print("-" * 50)
print("HASIL INFERENSI MODEL Random Forest")
print("-" * 50)
for teks, label in zip(test_texts, results):
    print(f"Teks  : {teks}")
    print(f"Hasil : {label.upper()}")
    print("-" * 50)

--------------------------------------------------
HASIL INFERENSI MODEL Random Forest
--------------------------------------------------
Teks  : Aplikasi Gojek sangat membantu saya memesan makanan dengan cepat!
Hasil : POSITIF
--------------------------------------------------
Teks  : Biasa saja, tidak ada yang istimewa dari update kali ini.
Hasil : NEGATIF
--------------------------------------------------
Teks  : Kecewa banget, aplikasinya sering error dan driver tidak datang.
Hasil : NEGATIF
--------------------------------------------------
Teks  : Aplikasi ini sangat jelek sekali
Hasil : NEGATIF
--------------------------------------------------
Teks  : Ada beberapa promo yang muncul di halaman utama tadi pagi.
Hasil : POSITIF
--------------------------------------------------


In [78]:
#untuk skenario 3
#teks yang akan di uji
test_texts = [
    "Aplikasi Gojek sangat membantu saya memesan makanan dengan cepat!",
    "Biasa saja, tidak ada yang istimewa dari update kali ini.",
    "Kecewa banget, aplikasinya sering error dan driver tidak datang.",
    "Aplikasi ini sangat jelek sekali",
    "Ada beberapa promo yang muncul di halaman utama tadi pagi."
]

# Transformasi teks ke bentuk numerik (Wajib pakai tfidf yang sudah di-fit sebelumnya)
X_new_tfidf = tfidf2.transform(test_texts)

# Prediksi menggunakan model SVM terbaik hasil RandomSearch/BayesSearch
y_pred_new = opt_rf2.predict(X_new_tfidf)

# Mapping hasil angka kembali ke label teks
label_map_inverse = {0: 'negatif', 1: 'netral', 2: 'positif'}
results = [label_map_inverse[pred] for pred in y_pred_new]

#menampilkan hasil
print("-" * 50)
print("HASIL INFERENSI MODEL Random Forest")
print("-" * 50)
for teks, label in zip(test_texts, results):
    print(f"Teks  : {teks}")
    print(f"Hasil : {label.upper()}")
    print("-" * 50)

--------------------------------------------------
HASIL INFERENSI MODEL Random Forest
--------------------------------------------------
Teks  : Aplikasi Gojek sangat membantu saya memesan makanan dengan cepat!
Hasil : POSITIF
--------------------------------------------------
Teks  : Biasa saja, tidak ada yang istimewa dari update kali ini.
Hasil : NETRAL
--------------------------------------------------
Teks  : Kecewa banget, aplikasinya sering error dan driver tidak datang.
Hasil : NEGATIF
--------------------------------------------------
Teks  : Aplikasi ini sangat jelek sekali
Hasil : NEGATIF
--------------------------------------------------
Teks  : Ada beberapa promo yang muncul di halaman utama tadi pagi.
Hasil : NETRAL
--------------------------------------------------
